In [1]:
import numpy as np
import pandas as pd
np.random.seed(42)

In [15]:

# Item catalog
items = pd.DataFrame({
    "item_id": range(15),
    "name": [
        "Chicken Biryani",
        "Mutton Biryani",
        "Paneer Biryani",
        "Salan",
        "Raita",
        "Chicken Tikka",
        "Paneer Tikka",
        "Gulab Jamun",
        "Rasgulla",
        "Coke",
        "Pepsi",
        "Masala Chaas",
        "Butter Naan",
        "Dal Makhani",
        "Veg Pulao"
    ],
    "category": [
        "main", "main", "main",
        "side", "side",
        "side", "side",
        "dessert", "dessert",
        "beverage", "beverage", "beverage",
        "side", "main", "main"
    ],
    "price": [
        300, 350, 250,
        80, 60,
        220, 200,
        90, 85,
        40, 40, 50,
        30, 180, 220
    ]
})

items

,item_id,name,category,price
0,0,Chicken Biryani,main,300
1,1,Mutton Biryani,main,350
2,2,Paneer Biryani,main,250
3,3,Salan,side,80
4,4,Raita,side,60
5,5,Chicken Tikka,side,220
6,6,Paneer Tikka,side,200
7,7,Gulab Jamun,dessert,90
8,8,Rasgulla,dessert,85
9,9,Coke,beverage,40


In [16]:
users = pd.DataFrame({
    "user_id": range(10),
    "name": [
        "Matt", "John", "Aisha", "Rahul", "Priya",
        "Arjun", "Sneha", "David", "Fatima", "Karan"
    ],
    "price_sensitivity": np.random.rand(10)
})

users

,user_id,name,price_sensitivity
0,0,Matt,0.395956
1,1,John,0.992638
2,2,Aisha,0.637256
3,3,Rahul,0.334244
4,4,Priya,0.250024
5,5,Arjun,0.209482
6,6,Sneha,0.401641
7,7,David,0.193266
8,8,Fatima,0.135186
9,9,Karan,0.364063


In [17]:
orders = []
order_id = 0

for _ in range(200):
    user = np.random.choice(users.user_id)
    
    # Always choose a main
    main_items = items[items.category == "main"]
    main = np.random.choice(main_items.item_id)
    
    order_items = [main]
    
    # 70% chance add side
    if np.random.rand() < 0.7:
        side_items = items[items.category == "side"]
        order_items.append(np.random.choice(side_items.item_id))
    
    # 50% chance dessert
    if np.random.rand() < 0.5:
        dessert_items = items[items.category == "dessert"]
        order_items.append(np.random.choice(dessert_items.item_id))
    
    # 40% chance beverage
    if np.random.rand() < 0.4:
        beverage_items = items[items.category == "beverage"]
        order_items.append(np.random.choice(beverage_items.item_id))
    
    for position, item in enumerate(order_items):
        orders.append({
            "order_id": order_id,
            "user_id": user,
            "position": position,
            "item_id": item
        })
    
    order_id += 1

orders = pd.DataFrame(orders)
orders.head()

,order_id,user_id,position,item_id
0,0,8,0,13
1,0,8,1,4
2,0,8,2,8
3,0,8,3,9
4,1,2,0,2


In [18]:
training_rows = []

for order_id, group in orders.groupby("order_id"):
    group = group.sort_values("position")
    cart_items = []
    
    for _, row in group.iterrows():
        if len(cart_items) > 0:
            training_rows.append({
                "user_id": row.user_id,
                "cart_items": cart_items.copy(),
                "target_item": row.item_id
            })
        cart_items.append(row.item_id)

training_df = pd.DataFrame(training_rows)
training_df.head()

,user_id,cart_items,target_item
0,8,[13],4
1,8,"[13, 4]",8
2,8,"[13, 4, 8]",9
3,2,[2],3
4,2,"[2, 3]",7


In [19]:
from collections import defaultdict

co_matrix = defaultdict(lambda: defaultdict(int))

for order_id, group in orders.groupby("order_id"):
    item_list = list(group.sort_values("position").item_id)
    
    for i in range(len(item_list)):
        for j in range(i+1, len(item_list)):
            co_matrix[item_list[i]][item_list[j]] += 1

co_prob = {}

for i in co_matrix:
    total = sum(co_matrix[i].values())
    co_prob[i] = {
        j: co_matrix[i][j] / total
        for j in co_matrix[i]
    }

In [20]:
ranking_data = []

for _, row in training_df.iterrows():
    user = row.user_id
    cart = row.cart_items
    positive = row.target_item
    
    last_item = cart[-1]
    candidates = list(co_prob.get(last_item, {}).keys())
    
    for candidate in candidates:
        cart_value = items[items.item_id.isin(cart)].price.sum()
        candidate_price = items.loc[
            items.item_id == candidate, "price"
        ].values[0]
        
        ranking_data.append({
            "user_id": user,
            "cart_size": len(cart),
            "cart_value": cart_value,
            "candidate_item": candidate,
            "candidate_price": candidate_price,
            "label": 1 if candidate == positive else 0
        })

ranking_df = pd.DataFrame(ranking_data)

ranking_df

,user_id,cart_size,cart_value,candidate_item,candidate_price,label
0,8,1,180,4,60,1
1,8,1,180,8,85,0
2,8,1,180,9,40,0
3,8,1,180,5,220,0
4,8,1,180,7,90,0
...,...,...,...,...,...,...
2382,1,2,330,8,85,0
2383,1,2,330,10,40,0
2384,1,2,330,7,90,0
2385,1,2,330,11,50,0


In [21]:
ranking_df["price_diff"] = (
    ranking_df["candidate_price"] -
    ranking_df["cart_value"] / ranking_df["cart_size"]
)

In [22]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = ranking_df.drop("label", axis=1)
y = ranking_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test)

params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 31
}

model = lgb.train(params, train_data, valid_sets=[test_data], num_boost_round=100)

preds = model.predict(X_test)
print("AUC:", roc_auc_score(y_test, preds))

[LightGBM] [Info] Number of positive: 259, number of negative: 1650
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000343 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 144
[LightGBM] [Info] Number of data points in the train set: 1909, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.135673 -> initscore=-1.851703
[LightGBM] [Info] Start training from score -1.851703
AUC: 0.6913873792270532


In [ ]:
# import lightgbm as lgb

# train_data = lgb.Dataset(X_train, label=y_train)
# test_data = lgb.Dataset(X_test, label=y_test)

# params = {
#     "objective": "binary",
#     "metric": "auc",
#     "learning_rate": 0.05,
#     "num_leaves": 31
# }

# model = lgb.train(
#     params,
#     train_data,
#     valid_sets=[test_data],
#     num_boost_round=100
# )

[LightGBM] [Info] Number of positive: 2355, number of negative: 43313
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001062 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 650
[LightGBM] [Info] Number of data points in the train set: 45668, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.051568 -> initscore=-2.911912
[LightGBM] [Info] Start training from score -2.911912


In [ ]:
# from sklearn.metrics import roc_auc_score

# preds = model.predict(X_test)
# auc = roc_auc_score(y_test, preds)

# print("AUC:", auc)

AUC: 0.490813323323547


In [ ]:
# ranking_df["pred"] = model.predict(X)

# precision_at_5 = ranking_df.sort_values("pred", ascending=False) \
#                             .head(5)["label"].mean()

# print("Precision@5:", precision_at_5)

Precision@5: 0.4


In [23]:
def recommend(user_id, cart_items, top_n=5):
    last_item = cart_items[-1]
    candidates = list(co_prob.get(last_item, {}).keys())
    
    rows = []
    
    for candidate in candidates:
        cart_value = items[items.item_id.isin(cart_items)].price.sum()
        candidate_price = items.loc[
            items.item_id == candidate, "price"
        ].values[0]
        
        rows.append({
            "user_id": user_id,
            "cart_size": len(cart_items),
            "cart_value": cart_value,
            "candidate_item": candidate,
            "candidate_price": candidate_price,
            "price_diff": candidate_price - cart_value / len(cart_items)
        })
    
    df = pd.DataFrame(rows)
    df["score"] = model.predict(df)
    
    df = df.sort_values("score", ascending=False).head(top_n)
    
    df = df.merge(items[["item_id", "name"]], 
                  left_on="candidate_item", 
                  right_on="item_id")
    
    return df[["name", "score"]]

In [24]:
recommend(user_id=1, cart_items=[0])  # Chicken Biryani

,name,score
0,Butter Naan,0.453830
1,Chicken Tikka,0.172684
2,Paneer Tikka,0.153764
3,Raita,0.076407
4,Gulab Jamun,0.068295
